# 📝 EDA·데이터 시각화 과제 LV3(통합) — 데이터셋 EDA 리포트

> 하나의 데이터셋을 **불러오기 → 집계 → 시각화 → 인사이트**의 순서로 처음부터 끝까지 분석하는 통합 문제입니다. 각 `### N단계` 셀에 그 단계에서 할 일이 적혀 있어요.

## 풀이 방법
1. 문제마다 **1단계에서 데이터를 한 번 불러와** 같은 `df` 로 끝까지 이어 분석합니다.
2. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채웁니다. 집계 단계는 아래 **자가채점 셀**로 확인하고, **시각화 단계는 자가채점 없이** 위 **완성 그래프(정답)** 와 같은 모양으로 그립니다.
3. 그래프는 결과 Axes 를 지정된 이름(`ax1`·`ax2`·`ax3`)에 저장하고 제목만 `set_title` 으로 다세요(축 이름은 자동).
4. 마지막 **인사이트 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

화이팅!

In [ ]:
# [제공 코드] 시각화 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('data/diamonds.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[수치 요약] describe()"); display(df.describe())

## 1. 다이아몬드 컷·색 품질 EDA 리포트
**배경**: `diamonds.csv` 는 다이아몬드 5000개의 무게·컷·색·가격입니다. 이번에는 **컷(cut)·색(color) 같은 품질 등급**이 어떻게 분포하고, 그 등급이 가격과 어떻게 이어지는지 집계·시각화로 탐색합니다.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 분석합니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원본 (5000, 10), `cut` 5종, `color` 7종 |
| 2단계 | D색 평균가 2999.74, J색 평균가 5755.88, 컷×색 평균 캐럿·개수 표 (각 5, 7), `color_summary` 4열(D색 가격범위 18202 · J색 고가비율 18.73 · D색 최빈 투명도 SI1) |
| 3단계 | countplot·히스토그램·산점도 3종 (제목·축 라벨 확인) |
| 4단계 | 인사이트 서술(3문장 이상) |

### 1단계 — 데이터 불러오기·구조 파악
`data/diamonds.csv` 를 `df` 로 불러오고, `df.shape` 와 `df['cut'].value_counts()`, `df['color'].value_counts()`, 그리고 `df['price'].describe()` 를 출력하세요. 원본은 `(5000, 10)` 이고, `cut` 은 5종(Fair·Good·Very Good·Premium·Ideal), `color` 는 7종(D~J) 입니다.

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape == (5000, 10)
assert df['cut'].nunique() == 5
assert df['color'].nunique() == 7
print("✅ 1단계 통과!")

### 2단계 — 집계 (groupby · pivot_table · crosstab · 사용자 정의 집계)
같은 `df` 로 세 가지 품질 집계를 만드세요.
- `price_by_color`: `color` 별 평균 `price`(색 등급별 평균 가격) — `groupby`
- `pv`: `cut`(행) × `color`(열) 의 평균 `carat`(평균 무게) — `pivot_table` (모양은 `(5, 7)`)
- `ct`: `cut`(행) × `color`(열) 의 **개수** 교차표 — `crosstab` (모양은 `(5, 7)`)
- `color_summary`: `color` 별로 아래 **세 열**을 가진 표 — **사용자 정의 집계**(결과 열 이름을 직접 붙이고, 소수 둘째 자리로 반올림)
  - `평균가` — `price` 의 평균  ·  `가격범위` — `price` 의 **최댓값 − 최솟값**(`lambda`)  ·  `고가비율` — `price` 가 **10000 이상**인 비율(%)  ·  `최빈_투명도` — `clarity` 의 **최빈값**(`mode()[0]`)
  - 앞의 셋은 수치형 집계, 마지막 하나는 **범주형 집계**입니다. 색 등급이 좋으면 투명도 등급도 좋은지 함께 보려는 것이죠.

(참고: D색 평균가 2999.74, J색 평균가 5755.88, Ideal·G 평균 캐럿 0.666, Ideal·G 개수 437, D색 가격범위 18202, J색 고가비율 18.73, D색 최빈 투명도 SI1, G색 최빈 투명도 VS2)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert round(float(price_by_color['D']), 2) == 2999.74
assert round(float(price_by_color['J']), 2) == 5755.88
assert pv.shape == (5, 7)
assert round(float(pv.loc['Ideal', 'G']), 3) == 0.666
assert ct.shape == (5, 7)
assert int(ct.loc['Ideal', 'G']) == 437
assert color_summary.shape == (7, 4), '색 7종 × 4열이 나와야 해요'
assert list(color_summary.columns) == ['평균가', '가격범위', '고가비율', '최빈_투명도'], '열 이름을 지문 그대로 써 주세요'
assert round(float(color_summary.loc['D', '평균가']), 2) == 2999.74
assert round(float(color_summary.loc['D', '가격범위']), 2) == 18202.0
assert color_summary.loc['D', '최빈_투명도'] == 'SI1', 'clarity 의 최빈값이에요'
assert color_summary.loc['G', '최빈_투명도'] == 'VS2'
assert round(float(color_summary.loc['J', '고가비율']), 2) == 18.73, '10000 이상인 비율(%)이에요 — 100을 곱했나요?'
print("✅ 2단계 통과!")

### 3단계 — 시각화 3종 (범주·분포·관계)
같은 `df` 로 세 그래프를 그리고, 각 결과 Axes 를 지정 이름에 저장한 뒤 제목을 다세요.
- `ax1`: **컷 등급별 개수 막대그래프** — `countplot(x='cut')` (범주)
- `ax2`: **무게(carat) 분포 히스토그램** — `histplot(x='carat', bins=30)` (분포)
- `ax3`: **무게-가격 산점도(색 등급별)** — `scatterplot(x='carat', y='price', hue='color')` (관계)
- 그래프가 겹치지 않도록 각 그래프 앞에 `plt.figure()` 를 호출하세요.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q1_s3_1.png" width="520"/>

<img src="images/과제/lv3_q1_s3_2.png" width="520"/>

<img src="images/과제/lv3_q1_s3_3.png" width="520"/>

In [ ]:
# 여기에 코드를 작성하세요

### 4단계 — 인사이트 (서술)
위 집계와 그래프를 근거로 **컷·색 품질 등급이 가격과 어떻게 이어지는지**를 **3문장 이상** 서술하세요.
- 그래프에서 **보이는 경향**을 말로 표현하세요 (예: 어떤 등급이 더 흔한지, 무게가 클수록 가격이 어떻게 되는지).

*(여기에 3문장 이상으로 인사이트를 서술하세요)*

## 2. 다이아몬드 가격 EDA 리포트
**배경**: `diamonds.csv` 는 다이아몬드 5000개의 무게·컷·색·가격입니다. 가격이 무엇에 좌우되는지 집계·시각화로 탐색합니다.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 분석합니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원본 (5000, 10), `price` 평균 3917.29, `carat` 평균 0.797 |
| 2단계 | Premium 평균가 4587.82, Ideal 평균가 3321.98, 컷×색 피벗 (5, 7), `ppc_by_cut` 캐럿당 가격(Fair 3550.40 · Premium 4201.88) |
| 3단계 | 히스토그램·산점도·히트맵 3종을 위 **완성 그래프**처럼 (자가채점 없음) |
| 4단계 | 인사이트 서술 |

### 1단계 — 데이터 불러오기·기초 파악
`data/diamonds.csv` 를 `df` 로 불러오고, `df.shape` 와 함께 `df['carat'].describe()`, `df['price'].describe()` 를 출력하세요. 원본은 `(5000, 10)`, `price` 평균은 약 3917.29, `carat` 평균은 약 0.797 입니다.

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape == (5000, 10)
assert round(float(df['price'].mean()), 2) == 3917.29
assert round(float(df['carat'].mean()), 3) == 0.797
print("✅ 1단계 통과!")

### 2단계 — 집계 (groupby · pivot_table · apply)
같은 `df` 로 두 집계를 만드세요.
- `price_by_cut`: `cut` 별 평균 `price` — `groupby`
- `pv`: `cut`(행) × `color`(열) 의 평균 `price` — `pivot_table` (모양은 `(5, 7)`)
- `ppc_by_cut`: `cut` 별 **'캐럿당 가격'의 평균** — 다이아 **한 개씩** `price ÷ carat` 을 구한 뒤 그 평균을 냅니다. 두 열을 함께 써야 하니 `apply` 로 만드세요(결과는 Series, 인덱스는 `cut`). 쓸 두 열만 골라 넘기면(`[['price', 'carat']]`) 경고 없이 깔끔합니다.
  - ⚠️ `price` **합계** ÷ `carat` **합계**로 계산하면 값이 달라집니다(Fair 3984.43). **행마다 나눈 뒤 평균**을 내세요.

(참고: Premium 평균가 4587.82, Ideal 평균가 3321.98, Ideal·G 조합 3387.32, 캐럿당 가격 Fair 3550.40 · Premium 4201.88)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert round(float(price_by_cut['Premium']), 2) == 4587.82
assert round(float(price_by_cut['Ideal']), 2) == 3321.98
assert pv.shape == (5, 7)
assert round(float(pv.loc['Ideal', 'G']), 2) == 3387.32
assert round(float(ppc_by_cut['Fair']), 2) == 3550.4, '행마다 price/carat 을 구한 뒤 평균을 내세요(합계로 나누면 3984.43 이 나옵니다)'
assert round(float(ppc_by_cut['Premium']), 2) == 4201.88
print("✅ 2단계 통과!")

### 3단계 — 시각화 3종 (분포·관계·집계 히트맵)
같은 `df`(와 2단계의 `pv`)로 세 그래프를 그리고, 각 결과 Axes 를 지정 이름에 저장한 뒤 제목을 다세요.
- `ax1`: **가격 분포 히스토그램** — `histplot(x='price', bins=30)` (분포)
- `ax2`: **무게-가격 산점도** — `scatterplot(x='carat', y='price', hue='cut')` (관계)
- `ax3`: **컷×색 평균가격 히트맵** — 2단계 `pv` 를 `heatmap(annot=True, fmt='.0f')` 로 (집계값 히트맵)
- 그래프가 겹치지 않도록 각 그래프 앞에 `plt.figure()` 를 호출하세요.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q2_s3_1.png" width="520"/>

<img src="images/과제/lv3_q2_s3_2.png" width="520"/>

<img src="images/과제/lv3_q2_s3_3.png" width="520"/>

In [ ]:
# 여기에 코드를 작성하세요

### 4단계 — 인사이트 (서술)
위 집계와 그래프를 근거로 **가격을 좌우하는 요인**을 **3문장 이상** 서술하세요.
- 그래프에서 **보이는 경향**을 말로 표현하세요 (예: 캐럿이 클수록 가격이 오르는 편).

*(여기에 3문장 이상으로 인사이트를 서술하세요)*